### Esse notebook tem como objetivo utilizar as principais funcoes do spark usando arquivos delta

In [2]:
from pyspark.sql import SparkSession
from delta import *
from pyspark.sql import functions as F

# Pacotes necessários (incluindo Delta)
packages = ",".join([
    "io.delta:delta-spark_2.12:3.2.0",
    "org.apache.hadoop:hadoop-aws:3.3.2",
    "com.amazonaws:aws-java-sdk-bundle:1.12.628"
])

builder = SparkSession.builder \
    .appName("DeltaLakeApp") \
    .config("spark.jars.packages", packages) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

print(f"✅ Spark {spark.version} com Delta Lake configurado!")

✅ Spark 3.5.0 com Delta Lake configurado!


In [3]:
spark

In [4]:
nam_path_s3 = "s3a://datalake/tmp_data/netflix_titles/"

### Criando tabela em delta

In [4]:
df_pyspark=spark.read.csv('s3a://datalake/raw_data/netflix_titles.csv', inferSchema=True, header=True, sep=',', quote='"', escape='"', multiLine=True)

In [5]:
df_pyspark.write.mode("overwrite").format("delta").option("path",nam_path_s3).saveAsTable('netflix_titles')

### Leitura de dados

In [5]:
df_delta  = spark.read.format("delta").load(nam_path_s3)

In [6]:
df_delta.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [7]:
df_delta.groupBy('type').agg(F.count('*')).show(10)

+-------+--------+
|   type|count(1)|
+-------+--------+
|TV Show|    3154|
|  Movie|    6652|
+-------+--------+



### Update dos dados

In [9]:
table = DeltaTable.forPath(spark,nam_path_s3)

In [11]:
table.update(
    condition="type = 'Movie'",
    set={"type": "'movie'"})

In [12]:
table.toDF().groupBy('type').agg(F.count('*')).show()

+-------+--------+
|   type|count(1)|
+-------+--------+
|TV Show|    2676|
|  movie|    6131|
+-------+--------+



In [13]:
table.toDF().count()

8807

### Update e insert de dados (Merge)

In [14]:
df_tmp=spark.read.csv('s3a://datalake/raw_data/netflix_titles_augmented.csv', inferSchema=True, header=True, sep=',', quote='"', escape='"', multiLine=True)

In [15]:
df_tmp.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [16]:
df_tmp.groupBy('type').agg(F.count('*')).show(10)

+-------+--------+
|   type|count(1)|
+-------+--------+
|TV Show|    3154|
|  Movie|    6653|
+-------+--------+



In [17]:
table = DeltaTable.forPath(spark, nam_path_s3)
table.alias("netflix_titles_x") \
    .merge(
        df_tmp.alias("netflix_titles_tmp"),
        "netflix_titles_x.show_id = netflix_titles_tmp.show_id",
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print("✅ Merge executado com sucesso!")

✅ Merge executado com sucesso!


In [18]:
table.toDF().groupBy('type').agg(F.count('*')).show()

+-------+--------+
|   type|count(1)|
+-------+--------+
|TV Show|    3154|
|  Movie|    6653|
+-------+--------+



In [19]:
table.toDF().count()

9807

### Delete dos Dados

In [20]:
table_delete = DeltaTable.forPath(spark,nam_path_s3)
table_delete.delete("show_id='s1'")

In [21]:
table_delete.toDF().select('show_id','title').filter(F.col('show_id')=='s1').show(10)

+-------+-----+
|show_id|title|
+-------+-----+
+-------+-----+



In [23]:
table_delete.toDF().select('show_id','title').orderBy('show_id').show(10, truncate=False)

+-------+----------------------------------+
|show_id|title                             |
+-------+----------------------------------+
|s10    |The Starling                      |
|s100   |On the Verge                      |
|s1000  |Stowaway                          |
|s1001  |Wild Dog                          |
|s1002  |Oloibiri                          |
|s1003  |Tell Me When                      |
|s1004  |Zero                              |
|s1005  |Izzy's Koala World                |
|s1006  |Keymon and Nani in Space Adventure|
|s1007  |Motu Patlu Dino Invasion          |
+-------+----------------------------------+
only showing top 10 rows



### Recuperação de Histórico

In [8]:
history_df = DeltaTable.forPath(spark,nam_path_s3)

In [9]:
fullHistoryDF = history_df.history() 

In [40]:
fullHistoryDF.select('version','timestamp','operation','userMetadata').orderBy(F.asc('timestamp')).show(10,truncate=False)

+-------+-------------------+---------------------------------+------------+
|version|timestamp          |operation                        |userMetadata|
+-------+-------------------+---------------------------------+------------+
|0      |2026-01-06 14:52:11|CREATE OR REPLACE TABLE AS SELECT|NULL        |
|1      |2026-01-06 14:53:21|UPDATE                           |NULL        |
|2      |2026-01-06 14:54:51|MERGE                            |NULL        |
|3      |2026-01-06 14:58:11|DELETE                           |NULL        |
+-------+-------------------+---------------------------------+------------+



In [32]:
df_delta_hist  = spark.read.format("delta").option('versionAsof',0).load(nam_path_s3)

In [33]:
df_delta_hist.groupBy('type').agg(F.count('*')).show()

+-------+--------+
|   type|count(1)|
+-------+--------+
|TV Show|    2676|
|  Movie|    6131|
+-------+--------+



In [34]:
df_delta_hist.agg(F.count('*')).show()

+--------+
|count(1)|
+--------+
|    8807|
+--------+



### drop de coluna

AnalysisException: Missing field diretor in table spark_catalog.default.netflix_titles with schema:
root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)
.; line 2 pos 4

In [43]:
spark.sql("DESCRIBE TABLE netflix_titles").show()

+------------+---------+-------+
|    col_name|data_type|comment|
+------------+---------+-------+
|     show_id|   string|   NULL|
|        type|   string|   NULL|
|       title|   string|   NULL|
|        cast|   string|   NULL|
|     country|   string|   NULL|
|  date_added|   string|   NULL|
|release_year|      int|   NULL|
|      rating|   string|   NULL|
|    duration|   string|   NULL|
|   listed_in|   string|   NULL|
| description|   string|   NULL|
+------------+---------+-------+



### Pendente
Criação de tabela temporaria,
Particionamento dos dados